# AutoScript MRC to HDF5 Debugging

Inspect every representation of one AutoScript 4D-STEM acquisition: raw MRC, SciFiReaders, manually written HDF5, reopened HDF5, and the DATA/Tiled result. Run the cells in order and stop where the data first looks wrong.

### Imports

In [ ]:
import json
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import mrcfile
import numpy as np
import tango
from SciFiReaders import MRCReader
from tiled.client import from_uri

%matplotlib ipympl

### Select one AutoScript MRC

Set `MRC_PATH` to a specific file or leave it as `None` to select the newest MRC. For the production DATA step, this path must also be readable by the DATA host.

In [ ]:
MRC_DIRECTORY = Path(r"Y:\AutoScript TEM")
MRC_PATH = None
MRC_INDEX = -1  # Use -5 for the first file in a five-angle MAPED series
EXPECTED_SCAN_SHAPE = None  # Example: (128, 128)

mrc_files = sorted(MRC_DIRECTORY.glob("*.mrc"), key=lambda path: path.stat().st_mtime_ns)
print("Recent MRC files:")
for index, path in enumerate(mrc_files[-10:], start=max(0, len(mrc_files) - 10)):
    print(index, path.name, path.stat().st_size, "bytes")
mrc_path = Path(MRC_PATH) if MRC_PATH else mrc_files[MRC_INDEX]
debug_directory = Path("outputs/mrc_debug")
debug_directory.mkdir(parents=True, exist_ok=True)
debug_h5_path = debug_directory / f"{mrc_path.stem}_manual.h5"

print("MRC:     ", mrc_path)
print("Size:    ", mrc_path.stat().st_size, "bytes")
print("Debug H5:", debug_h5_path)

## 1. Read the raw MRC

This is the file before SciFiReaders reshapes the flat diffraction-frame sequence into scan coordinates.

In [ ]:
raw_mrc = mrcfile.mmap(str(mrc_path), mode="r", permissive=True)
raw_data = raw_mrc.data
extended_header = raw_mrc.indexed_extended_header

print("Raw shape: ", raw_data.shape)
print("Raw dtype: ", raw_data.dtype)
print("MRC nx, ny, nz:", int(raw_mrc.header.nx), int(raw_mrc.header.ny), int(raw_mrc.header.nz))
print("MRC mode:   ", int(raw_mrc.header.mode))
print("MRC mapc, mapr, maps:", int(raw_mrc.header.mapc), int(raw_mrc.header.mapr), int(raw_mrc.header.maps))
print("Value range:", float(np.min(raw_data)), float(np.max(raw_data)))

### Display raw diffraction frames

These frames are displayed before any scan reshaping. Log intensity makes weak diffraction features visible.

In [ ]:
raw_indices = [0, raw_data.shape[0] // 2, raw_data.shape[0] - 1]
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, index in zip(axes, raw_indices):
    frame = np.asarray(raw_data[index])
    ax.imshow(np.log1p(np.clip(frame, 0, None)), cmap="magma")
    ax.set_title(f"raw frame {index}")
    ax.axis("off")
fig.tight_layout()

### Average all raw diffraction frames

If individual frames are noise but the average contains diffraction features, the acquisition has insufficient per-frame signal. If the average is also featureless, the camera did not receive a stable diffraction pattern.

In [ ]:
mean_pattern = np.asarray(raw_data.mean(axis=0, dtype=np.float64))
frame_means = np.asarray(raw_data.mean(axis=(-2, -1), dtype=np.float64))
display_min, display_max = np.percentile(mean_pattern, [1, 99.9])
peak_row, peak_column = np.unravel_index(np.argmax(mean_pattern), mean_pattern.shape)
background = np.concatenate([mean_pattern[:16].ravel(), mean_pattern[-16:].ravel(), mean_pattern[:, :16].ravel(), mean_pattern[:, -16:].ravel()])
peak_snr = (mean_pattern.max() - np.median(background)) / max(np.std(background), 1e-12)

print("Mean-pattern percentiles:", np.percentile(mean_pattern, [0, 1, 50, 99, 99.9, 100]))
print("Brightest mean pixel:    ", (peak_row, peak_column))
print("Peak/background SNR:    ", float(peak_snr))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mean_pattern, cmap="magma", vmin=display_min, vmax=display_max)
axes[0].set_title("mean pattern, linear")
axes[1].imshow(np.log1p(np.clip(mean_pattern, 0, None)), cmap="magma")
axes[1].set_title("mean pattern, log")
axes[2].plot(frame_means)
axes[2].set_title("mean counts per raw frame")
axes[2].set_xlabel("raw frame index")
for ax in axes[:2]:
    ax.axis("off")
fig.tight_layout()

## 2. Inspect the AutoScript extended header

SciFiReaders derives the scan shape from four extended-header fields. Incorrect bounds, swapped fields, or an interrupted scan can make the reshape look wrong.

In [ ]:
print("Extended-header fields:")
print(extended_header.dtype.names)

scan_labels = ["Scan size right", "Scan size left", "Scan size top", "Scan size bottom"]
scan_limits = {label: np.unique(extended_header[label]).tolist() for label in scan_labels}
for label, values in scan_limits.items():
    print(f"{label:18s}: {values}")

orientation_labels = ["Scan rotation", "Diffraction pattern rotation", "Image rotation", "Beam center X pixel", "Beam center Y pixel", "Readout area left", "Readout area top", "Readout area right", "Readout area bottom"]
acquisition_labels = ["Camera name", "Detector commercial name", "Dwell time", "Integration time", "Ceta frames summed", "Direct detector electron counting", "HT", "Projection mode", "Camera length"]
print("\nOrientation and camera metadata:")
for label in orientation_labels + acquisition_labels:
    if label in extended_header.dtype.names:
        print(f"{label:30s}: {np.unique(extended_header[label]).tolist()}")

right, left, top, bottom = [float(np.unique(extended_header[label])[0]) for label in scan_labels]
scan_shape_from_header = (int(abs(top - bottom)), int(abs(right - left)))
reshape_shape = (*scan_shape_from_header, *raw_data.shape[-2:])

print("Header scan shape:  ", scan_shape_from_header)
print("Raw frame count:    ", raw_data.shape[0])
print("Expected frame count:", int(np.prod(scan_shape_from_header)))
print("4D reshape target:  ", reshape_shape)
if EXPECTED_SCAN_SHAPE is not None:
    print("Matches requested scan:", tuple(EXPECTED_SCAN_SHAPE) == scan_shape_from_header)

### Reproduce the raw reshape

This is the exact C-order reshape used by SciFiReaders. No interpolation or intensity conversion occurs.

In [ ]:
if raw_data.shape[0] != np.prod(scan_shape_from_header):
    raise ValueError("The MRC frame count does not match the scan shape in its extended header")

raw_4d = raw_data.reshape(reshape_shape)
raw_virtual = np.asarray(raw_data.sum(axis=(-2, -1), dtype=np.float64)).reshape(scan_shape_from_header)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(raw_virtual, cmap="gray")
ax.set_title("Raw MRC virtual bright field")
ax.set_xlabel("scan axis 1")
ax.set_ylabel("scan axis 0")
fig.tight_layout()

## 3. Read with SciFiReaders

This is the same reader used by `DATA.copy_and_register_remote_file`.

In [ ]:
channels = MRCReader(str(mrc_path)).read()
scifi_data = channels["Channel_000"]

print(scifi_data)
print("SciFiReaders shape:", scifi_data.shape)
print("SciFiReaders dtype:", scifi_data.dtype)
print("SciFiReaders chunks:", scifi_data.chunks)
print("SciFiReaders data type:", scifi_data.data_type)
print("Metadata fields:", list(scifi_data.original_metadata))
for label in scan_labels + orientation_labels + acquisition_labels + ["Pixel size X", "Pixel size Y"]:
    if label in scifi_data.original_metadata:
        print(f"{label:30s}: {np.asarray(scifi_data.original_metadata[label]).tolist()}")

### Display SciFiReaders diffraction frames

In [ ]:
scan_positions = [(0, 0), (scifi_data.shape[0] // 2, scifi_data.shape[1] // 2), (scifi_data.shape[0] - 1, scifi_data.shape[1] - 1)]
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (row, column) in zip(axes, scan_positions):
    frame = np.asarray(scifi_data[row, column].compute())
    ax.imshow(np.log1p(np.clip(frame, 0, None)), cmap="magma")
    ax.set_title(f"SciFi [{row}, {column}]")
    ax.axis("off")
fig.tight_layout()

### Verify the raw-frame mapping

For C-order reshaping, scan position `(row, column)` must equal raw frame `row * scan_width + column`.

In [ ]:
for row, column in scan_positions:
    raw_index = row * scifi_data.shape[1] + column
    raw_frame = np.asarray(raw_data[raw_index])
    scifi_frame = np.asarray(scifi_data[row, column].compute())
    print((row, column), "-> raw", raw_index, "equal:", np.array_equal(raw_frame, scifi_frame), "max abs diff:", float(np.max(np.abs(raw_frame.astype(np.float64) - scifi_frame.astype(np.float64)))))

### Compare the complete scan-space intensity map

In [ ]:
scifi_virtual = np.asarray(scifi_data.sum(axis=(2, 3)).compute())
virtual_difference = scifi_virtual.astype(np.float64) - raw_virtual

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(raw_virtual, cmap="gray")
axes[0].set_title("raw reshape")
axes[1].imshow(scifi_virtual, cmap="gray")
axes[1].set_title("SciFiReaders")
axes[2].imshow(virtual_difference, cmap="coolwarm")
axes[2].set_title(f"difference, max={np.max(np.abs(virtual_difference)):.3g}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()

### Diffraction-orientation candidates

Use a recognizable asymmetric diffraction pattern to check whether the camera axes need transposition or reflection.

In [ ]:
row, column = scan_positions[1]
reference_frame = np.asarray(scifi_data[row, column].compute())
diffraction_views = {
    "as read": reference_frame,
    "transpose": reference_frame.T,
    "flip axis 0": np.flip(reference_frame, axis=0),
    "flip axis 1": np.flip(reference_frame, axis=1),
    "rotate +90": np.rot90(reference_frame, 1),
    "rotate -90": np.rot90(reference_frame, -1),
}
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, (name, image) in zip(axes.flat, diffraction_views.items()):
    ax.imshow(np.log1p(np.clip(image, 0, None)), cmap="magma")
    ax.set_title(name)
    ax.axis("off")
fig.tight_layout()

## 4. Write the HDF5 manually

This reproduces the DATA conversion one diffraction frame at a time.

In [ ]:
request_scan_shape = list(EXPECTED_SCAN_SHAPE or scifi_data.shape[:2])
with h5py.File(debug_h5_path, "w", track_order=True) as h5:
    dataset = h5.create_dataset("stem_data", shape=scifi_data.shape, dtype=scifi_data.dtype, chunks=True)
    for row in range(scifi_data.shape[0]):
        for column in range(scifi_data.shape[1]):
            dataset[row, column] = scifi_data[row, column].compute()
    dataset.attrs["acquisition_type"] = "stem_data"
    dataset.attrs["detector"] = "BM-Ceta"
    dataset.attrs["source_format"] = "MRC"
    dataset.attrs["source_file"] = str(mrc_path)
    dataset.attrs["data_type"] = str(scifi_data.data_type)
    dataset.attrs["scan_shape"] = json.dumps(request_scan_shape)
    h5.attrs["source_mrc_metadata_json"] = json.dumps(scifi_data.original_metadata, default=lambda value: value.tolist() if hasattr(value, "tolist") else str(value))

print(debug_h5_path, debug_h5_path.stat().st_size, "bytes")

### Inspect the HDF5 structure and metadata

In [ ]:
with h5py.File(debug_h5_path, "r") as h5:
    print("Root keys:", list(h5))
    print("Root attrs:", dict(h5.attrs))
    print("Dataset shape:", h5["stem_data"].shape)
    print("Dataset dtype:", h5["stem_data"].dtype)
    print("Dataset chunks:", h5["stem_data"].chunks)
    print("Dataset attrs:", dict(h5["stem_data"].attrs))

### Reopen and display the HDF5

If this differs from the SciFiReaders view, the error occurred during HDF5 writing.

In [ ]:
with h5py.File(debug_h5_path, "r") as h5:
    h5_virtual = np.empty(h5["stem_data"].shape[:2], dtype=np.float64)
    for row in range(h5["stem_data"].shape[0]):
        for column in range(h5["stem_data"].shape[1]):
            h5_virtual[row, column] = np.sum(h5["stem_data"][row, column], dtype=np.float64)
    h5_frames = [h5["stem_data"][row, column] for row, column in scan_positions]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(h5_virtual, cmap="gray")
axes[0].set_title("HDF5 virtual BF")
for ax, frame, position in zip(axes[1:], h5_frames, scan_positions):
    ax.imshow(np.log1p(np.clip(frame, 0, None)), cmap="magma")
    ax.set_title(f"HDF5 {position}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()

### Verify every HDF5 diffraction frame numerically

In [ ]:
mismatched_frames = 0
maximum_absolute_error = 0.0
with h5py.File(debug_h5_path, "r") as h5:
    for row in range(scifi_data.shape[0]):
        for column in range(scifi_data.shape[1]):
            source_frame = np.asarray(scifi_data[row, column].compute())
            h5_frame = h5["stem_data"][row, column]
            mismatched_frames += int(not np.array_equal(source_frame, h5_frame))
            maximum_absolute_error = max(maximum_absolute_error, float(np.max(np.abs(source_frame.astype(np.float64) - h5_frame.astype(np.float64)))))

print("Mismatched frames: ", mismatched_frames)
print("Maximum abs error:", maximum_absolute_error)
print("Virtual image equal:", np.array_equal(scifi_virtual, h5_virtual))

## 5. Run the production DATA conversion

This creates and registers a second HDF5 using `DATA.copy_and_register_remote_file`. Run this section only when the DATA host can read `mrc_path`.

In [ ]:
DB_HOST = "10.46.217.241"
DB_PORT = 9094
os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

data = tango.DeviceProxy("asyncroscopy/data/default")
data.set_timeout_millis(3_600_000)
data.ping()
tiled_config = json.loads(data.get_config())
client = from_uri(tiled_config["uri"])
print(tiled_config)

In [ ]:
production_request = {
    "source_path": str(mrc_path),
    "detector": "BM-Ceta",
    "scan_shape": request_scan_shape,
}
production_key = data.copy_and_register_remote_file(json.dumps(production_request))
print("Registered key:", production_key)

### Read the registered data through Tiled

Only three diffraction frames are transferred. Their exact comparison determines whether registration or Tiled changed the values.

In [ ]:
tiled_node = client[production_key]["stem_data"]
print("Tiled shape:", tiled_node.shape)
print("Tiled metadata:", dict(tiled_node.metadata))

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (row, column) in zip(axes, scan_positions):
    tiled_frame = tiled_node.read(slice=np.s_[row, column, :, :])
    h5_frame = h5_frames[scan_positions.index((row, column))]
    print((row, column), "Tiled equals HDF5:", np.array_equal(tiled_frame, h5_frame), "max abs diff:", float(np.max(np.abs(tiled_frame.astype(np.float64) - h5_frame.astype(np.float64)))))
    ax.imshow(np.log1p(np.clip(tiled_frame, 0, None)), cmap="magma")
    ax.set_title(f"Tiled [{row}, {column}]")
    ax.axis("off")
fig.tight_layout()

## Interpretation

- Raw frames wrong: acquisition or MRC writing problem.
- Raw frames correct but header-derived scan map wrong: scan bounds or frame-order problem.
- Raw reshape and SciFiReaders differ: reader problem.
- SciFiReaders correct but manual HDF5 differs: HDF5 conversion problem.
- Manual HDF5 correct but Tiled frames differ: registration or serving problem.
- Every numerical comparison passes but the result still looks wrong: compare the recorded scan, image, and diffraction rotations against the displayed data.

### Close the raw MRC

In [ ]:
raw_mrc.close()